# LSTM GPU Training for Predictive Maintenance

Notebook ini berisi:
1. Download data dari Google Drive
2. Preprocessing data
3. Model LSTM dengan CuPy (GPU acceleration)
4. Training dan Evaluasi

## 1. Setup & Install Dependencies

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install cupy-cuda12x gdown pandas scikit-learn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
import os

# Check GPU availability
try:
    import cupy as cp
    GPU_AVAILABLE = True
    print("✅ CuPy detected - Using GPU acceleration")
    print(f"GPU: {cp.cuda.runtime.getDeviceCount()} device(s) available")
except ImportError:
    import numpy as cp
    GPU_AVAILABLE = False
    print("⚠️ CuPy not found - Falling back to NumPy (CPU)")

## 2. Download Data dari Google Drive

Data folder: https://drive.google.com/drive/u/0/folders/1FTEGLqRHNLBD5Xbbkub2fmcB8xGJOtZF

In [ ]:
# Download data dari Google Drive (uncomment untuk download)
# import gdown
# 
# GDRIVE_FOLDER_ID = "1FTEGLqRHNLBD5Xbbkub2fmcB8xGJOtZF"
# DATA_DIR = "../data"
# 
# os.makedirs(DATA_DIR, exist_ok=True)
# gdown.download_folder(id=GDRIVE_FOLDER_ID, output=DATA_DIR, quiet=False)

In [ ]:
# Paths
DATA_DIR = "../data"
INPUT_FILE = os.path.join(DATA_DIR, "labeled_dataset.csv")

# Check if data exists
if os.path.exists(INPUT_FILE):
    print(f"✅ Data found: {INPUT_FILE}")
else:
    print(f"❌ Data not found: {INPUT_FILE}")
    print("Please download data from Google Drive first!")

## 3. Preprocessing Data

In [ ]:
# Hyperparameters
SEQUENCE_LENGTH = 5  # Number of rows to look back

def preprocess_data(filepath, sequence_length=5):
    """
    Preprocess raw data into sequences for LSTM.
    """
    print("=" * 60)
    print("PREPROCESSING FOR LSTM")
    print("=" * 60)
    
    # 1. Load data
    print("\n[1/5] Loading data...")
    df = pd.read_csv(filepath)
    print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
    print(f"Columns: {list(df.columns)}")
    
    # 2. Select features
    print("\n[2/5] Selecting features...")
    feature_cols = ['Service Type', 'Service Name', 'Type', 'Status', 
                    'SLA (minutes)', 'Month']
    label_col = 'label'
    
    df_features = df[feature_cols].copy()
    y = df[label_col].values
    
    # 3. Handle missing values and encode
    print("\n[3/5] Encoding categorical features...")
    encoders = {}
    categorical_cols = ['Service Type', 'Service Name', 'Type', 'Status', 'Month']
    
    for col in categorical_cols:
        df_features[col] = df_features[col].fillna('Unknown')
        le = LabelEncoder()
        df_features[col] = le.fit_transform(df_features[col].astype(str))
        encoders[col] = le
    
    # Fill numerical NaN with median
    df_features['SLA (minutes)'] = df_features['SLA (minutes)'].fillna(
        df_features['SLA (minutes)'].median()
    )
    
    # 4. Scale features
    print("\n[4/5] Scaling features...")
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(df_features.values)
    
    print(f"Feature shape: {X_scaled.shape}")
    
    # 5. Create sequences
    print("\n[5/5] Creating sequences...")
    X_sequences = []
    y_sequences = []
    
    for i in range(len(X_scaled) - sequence_length):
        X_sequences.append(X_scaled[i:i + sequence_length])
        y_sequences.append(y[i + sequence_length])
    
    X_sequences = np.array(X_sequences)
    y_sequences = np.array(y_sequences)
    
    print(f"\nSequences shape: {X_sequences.shape}")
    print(f"Labels shape: {y_sequences.shape}")
    print(f"Label distribution: 0={np.sum(y_sequences==0)}, 1={np.sum(y_sequences==1)}")
    
    return X_sequences, y_sequences, scaler, encoders

In [ ]:
# Run preprocessing
X, y, scaler, encoders = preprocess_data(INPUT_FILE, SEQUENCE_LENGTH)

## 4. LSTM Model (CuPy GPU-Accelerated)

In [ ]:
class LSTMModelGPU:
    """
    CuPy-accelerated LSTM Model for GPU Training.
    Drop-in replacement for the NumPy version.
    """
    
    def __init__(self, input_size, hidden_size, output_size=1):
        """
        GPU-accelerated LSTM for Tabular Data
        """
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        z_dim = hidden_size + input_size
        
        # Parameter Initialization
        self.params = {}
        for gate in ['f', 'i', 'c', 'o']:
            self.params[f'W{gate}'] = cp.random.randn(hidden_size, z_dim).astype(cp.float32) * 0.1
            self.params[f'b{gate}'] = cp.zeros((hidden_size, 1), dtype=cp.float32)
        
        self.params['Wy'] = cp.random.randn(output_size, hidden_size).astype(cp.float32) * 0.1
        self.params['by'] = cp.zeros((output_size, 1), dtype=cp.float32)
        
        self.grads = {}
        self.reset_gradients()

    def reset_gradients(self):
        for key in self.params:
            self.grads[f'd{key}'] = cp.zeros_like(self.params[key])

    def sigmoid(self, x):
        return 1 / (1 + cp.exp(-cp.clip(x, -500, 500)))

    def tanh(self, x):
        return cp.tanh(x)

    def forward_step(self, x_t, h_prev, c_prev):
        z = cp.row_stack((h_prev, x_t))
        
        f_t = self.sigmoid(cp.dot(self.params['Wf'], z) + self.params['bf'])
        i_t = self.sigmoid(cp.dot(self.params['Wi'], z) + self.params['bi'])
        c_bar = self.tanh(cp.dot(self.params['Wc'], z) + self.params['bc'])
        
        c_t = f_t * c_prev + i_t * c_bar
        
        o_t = self.sigmoid(cp.dot(self.params['Wo'], z) + self.params['bo'])
        h_t = o_t * self.tanh(c_t)
        
        y_pred = self.sigmoid(cp.dot(self.params['Wy'], h_t) + self.params['by'])
        
        cache = (z, f_t, i_t, c_bar, c_t, o_t, h_t, c_prev, h_prev)
        return h_t, c_t, y_pred, cache

    def backward_step(self, dy, dh_next, dc_next, cache):
        z, f, i, c_bar, c, o, h, c_prev, h_prev = cache
        
        self.grads['dWy'] += cp.dot(dy, h.T)
        self.grads['dby'] += dy
        
        dh = cp.dot(self.params['Wy'].T, dy) + dh_next
        
        do = dh * self.tanh(c)
        da_o = do * o * (1 - o)
        self.grads['dWo'] += cp.dot(da_o, z.T)
        self.grads['dbo'] += da_o
        
        dc = dh * o * (1 - self.tanh(c)**2) + dc_next
        
        dc_bar = dc * i
        da_c = dc_bar * (1 - c_bar**2)
        self.grads['dWc'] += cp.dot(da_c, z.T)
        self.grads['dbc'] += da_c
        
        di = dc * c_bar
        da_i = di * i * (1 - i)
        self.grads['dWi'] += cp.dot(da_i, z.T)
        self.grads['dbi'] += da_i
        
        df = dc * c_prev
        da_f = df * f * (1 - f)
        self.grads['dWf'] += cp.dot(da_f, z.T)
        self.grads['dbf'] += da_f
        
        dz = (cp.dot(self.params['Wf'].T, da_f) + 
              cp.dot(self.params['Wi'].T, da_i) + 
              cp.dot(self.params['Wc'].T, da_c) + 
              cp.dot(self.params['Wo'].T, da_o))
        
        dh_prev = dz[:self.hidden_size, :]
        dc_prev = f * dc
        
        return dh_prev, dc_prev

    def update_parameters(self, lr):
        for key in self.params:
            self.params[key] -= lr * self.grads[f'd{key}']

    def train(self, X, y, epochs=10, lr=0.01, print_every=1):
        """
        Main Training Loop (GPU-accelerated)
        """
        # Convert to GPU arrays
        X_gpu = cp.asarray(X, dtype=cp.float32)
        y_gpu = cp.asarray(y, dtype=cp.float32)
        
        history = {'loss': []}
        
        for epoch in range(epochs):
            loss_history = []
            for i in range(len(X_gpu)):
                h_prev = cp.zeros((self.hidden_size, 1), dtype=cp.float32)
                c_prev = cp.zeros((self.hidden_size, 1), dtype=cp.float32)
                self.reset_gradients()
                
                caches = []
                x_sequence = X_gpu[i]
                y_true = y_gpu[i]

                # Forward Pass
                for t in range(len(x_sequence)):
                    x_t = x_sequence[t].reshape(-1, 1)
                    h_prev, c_prev, y_pred, cache = self.forward_step(x_t, h_prev, c_prev)
                    caches.append(cache)
                
                # Loss (Binary Cross Entropy)
                loss = - (y_true * cp.log(y_pred + 1e-9) + (1 - y_true) * cp.log(1 - y_pred + 1e-9))
                loss_history.append(float(cp.squeeze(loss)))
                
                # Backward Pass (BPTT)
                dy = y_pred - y_true
                dh_next = cp.zeros_like(h_prev)
                dc_next = cp.zeros_like(c_prev)
                
                for t in reversed(range(len(x_sequence))):
                    step_dy = dy if t == len(x_sequence) - 1 else cp.zeros_like(dy)
                    dh_next, dc_next = self.backward_step(step_dy, dh_next, dc_next, caches[t])
                
                self.update_parameters(lr)
            
            # Sync GPU before printing
            if GPU_AVAILABLE:
                cp.cuda.Stream.null.synchronize()
            
            avg_loss = sum(loss_history) / len(loss_history)
            history['loss'].append(avg_loss)
            
            if (epoch + 1) % print_every == 0:
                print(f"Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}")
        
        return history
    
    def predict(self, X):
        """Generate predictions (returns NumPy array)"""
        X_gpu = cp.asarray(X, dtype=cp.float32)
        predictions = []
        
        for i in range(len(X_gpu)):
            h = cp.zeros((self.hidden_size, 1), dtype=cp.float32)
            c = cp.zeros((self.hidden_size, 1), dtype=cp.float32)
            
            for t in range(len(X_gpu[i])):
                x_t = X_gpu[i][t].reshape(-1, 1)
                h, c, y_pred, _ = self.forward_step(x_t, h, c)
            
            predictions.append(float(cp.squeeze(y_pred)))
        
        return np.array(predictions)

## 5. Train/Test Split

In [ ]:
# Hyperparameters
HIDDEN_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 0.01
TEST_SPLIT = 0.2

# Split data
split_idx = int(len(X) * (1 - TEST_SPLIT))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"\nInput size: {X.shape[2]}")
print(f"Sequence length: {X.shape[1]}")

## 6. Training

In [ ]:
print("=" * 60)
print("LSTM TRAINING" + (" (GPU)" if GPU_AVAILABLE else " (CPU)"))
print("=" * 60)

# Create model
input_size = X.shape[2]
model = LSTMModelGPU(input_size=input_size, hidden_size=HIDDEN_SIZE)

print(f"\nModel Config:")
print(f"  - Input size: {input_size}")
print(f"  - Hidden size: {HIDDEN_SIZE}")
print(f"  - Output size: 1 (binary classification)")
print(f"  - Learning rate: {LEARNING_RATE}")
print(f"  - Epochs: {EPOCHS}")

# Train
print("\nTraining...")
history = model.train(X_train, y_train, epochs=EPOCHS, lr=LEARNING_RATE, print_every=10)

In [ ]:
# Plot training loss
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(history['loss'])
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

## 7. Evaluation

In [ ]:
print("Generating predictions...")
probabilities = model.predict(X_test)
predictions = (probabilities >= 0.5).astype(int)

# Accuracy
accuracy = np.mean(predictions == y_test)
print(f"\n✅ Test Accuracy: {accuracy:.4f}")

In [ ]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=['Normal', 'Anomaly']))

In [ ]:
# Confusion Matrix
import seaborn as sns

cm = confusion_matrix(y_test, predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 8. Save Results

In [ ]:
# Save predictions to CSV
results_df = pd.DataFrame({
    'actual': y_test,
    'predicted': predictions,
    'probability': probabilities
})

output_file = os.path.join(DATA_DIR, 'predictions_gpu.csv')
results_df.to_csv(output_file, index=False)
print(f"✅ Results saved to: {output_file}")

In [ ]:
# Preview results
results_df.head(10)